# Convert Lumen to GGUF Q4_K_M
Runs on CPU — no GPU needed. Takes ~10 minutes.

In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Install dependencies
!pip install -q transformers huggingface_hub sentencepiece
!apt-get install -y cmake build-essential

In [ ]:
# 3. Clone llama.cpp and build quantize tool
!git clone https://github.com/ggerganov/llama.cpp /content/llama.cpp
!cd /content/llama.cpp && cmake -B build -DLLAMA_CURL=OFF && cmake --build build --config Release -j$(nproc) --target llama-quantize
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

HF_TOKEN   = ''
MODEL_PATH = '/content/drive/MyDrive/Lumen/lumen-merged'
GGUF_F16   = '/content/lumen-f16.gguf'
BASE       = 'meta-llama/Llama-3.1-8B-Instruct'

f1 = hf_hub_download(BASE, 'tokenizer.json', token=HF_TOKEN)
f2 = hf_hub_download(BASE, 'tokenizer_config.json', token=HF_TOKEN)
f3 = hf_hub_download(BASE, 'special_tokens_map.json', token=HF_TOKEN)
shutil.copy(f1, os.path.join(MODEL_PATH, 'tokenizer.json'))
shutil.copy(f2, os.path.join(MODEL_PATH, 'tokenizer_config.json'))
shutil.copy(f3, os.path.join(MODEL_PATH, 'special_tokens_map.json'))
print('Tokenizer files replaced.')

!python /content/llama.cpp/convert_hf_to_gguf.py {MODEL_PATH} --outfile {GGUF_F16} --outtype f16

if os.path.exists(GGUF_F16):
    print('F16 done. Size: ' + str(round(os.path.getsize(GGUF_F16)/1e9, 1)) + ' GB')
else:
    print('ERROR: file not created.')

In [ ]:
# 5. Quantize to Q4_K_M (~4.5GB)
GGUF_Q4 = '/content/drive/MyDrive/Lumen/lumen-q4_k_m.gguf'

!/content/llama.cpp/build/bin/llama-quantize {GGUF_F16} {GGUF_Q4} Q4_K_M

print('Quantization done.')

In [ ]:
# 6. Upload GGUF to HuggingFace
from huggingface_hub import HfApi

HF_TOKEN = ''  # paste your HF write token
api = HfApi(token=HF_TOKEN)

api.upload_file(
    path_or_fileobj=GGUF_Q4,
    path_in_repo='lumen-q4_k_m.gguf',
    repo_id='RavikxxBGamin/Lumen',
    repo_type='model',
)

print('Uploaded to HuggingFace!')